# NLP Project — LIME and SHAP Explainability Viewer

This notebook provides a premium, interactive interface to explore **LIME** and **SHAP** explanations generated for the various fine-tuned sentiment/emotion classification models in the project.

### Features:
1. **Dynamic Model Discovery**: Automatically scans the workspace directories recursively and detects any fine-tuned model results containing LIME/SHAP explanations.
2. **Multi-Format Support**:
   - **Format A (Summary-Based)**: Reads JSON summaries, extracts example indices, loads raw text from test sets, preprocesses on-the-fly, highlights words, and renders plots and iframe HTML reports.
   - **Format B (Static Image-Based)**: Reads folders containing pre-rendered LIME/SHAP PNG charts and iframe HTML files directly. Visualizes them side-by-side cleanly without dataset loader overhead.
3. **Collapsible Dropdowns**: Displays both the preprocessed text (with feature importance highlights) and the raw text before preprocessing using standard, responsive `<details>` HTML blocks.
4. **Dual Interactive Highlights**: Wraps top words with tooltips displaying LIME (green/red) and SHAP (blue/orange) weights.
5. **Side-by-Side Plots**: Generates comparative horizontal bar charts for the top 10 features according to LIME and SHAP.
6. **Safe HTML Integration**: Embeds the interactive LIME HTML report inside an isolated `<iframe>` using the `srcdoc` attribute to prevent CSS conflicts.
7. **Linked Widget Slider & Textbox**: Allows seamless iteration and direct selection of explained test samples.

In [1]:
import sys
from pathlib import Path

# Add project root and language shared directories to system path dynamically
notebook_dir = Path(".").resolve()
project_root = notebook_dir.parent

for path_to_add in [
    project_root,
    project_root / "shared" / "migrated",
    project_root / "1_English" / "shared",
    project_root / "2_Romanian" / "shared"
]:
    if str(path_to_add) not in sys.path:
        sys.path.insert(0, str(path_to_add))

import json
import re
import html
import pickle
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, HTML, clear_output
import ipywidgets as widgets

print("Setup complete. Imports successful.")

Setup complete. Imports successful.


In [2]:
def discover_model_registry(root_dir):
    """
    Walks the project directory recursively to discover Results directories
    containing LIME/SHAP explanations, extracting model names and autodetecting configurations.
    Handles both Format A (JSON summary-based) and Format B (static PNG/HTML image-based) runs.
    """
    registry = {}
    root_path = Path(root_dir)
    
    # 1. Discover Format A models
    for results_dir in root_path.rglob("Results"):
        lime_sum_path = results_dir / "lime_explanations" / "lime_summary.json"
        
        if lime_sum_path.exists():
            pipeline_dir = results_dir.parent
            model_name = pipeline_dir.name
            
            pipeline_dir_str = str(pipeline_dir)
            if "1_English" in pipeline_dir_str:
                dataset_name = "imdb"
                text_col = "review"
                label_col = "label"
                class_names = ["negative", "positive"]
                loader_module = "dataset_en"
                preprocess_module = "preprocessing_en"
                preprocess_fn = "preprocess_batch"
            elif "1_BERT_Sentiment_Articles" in pipeline_dir_str:
                dataset_name = "ro_articles"
                text_col = "article"
                label_col = "label"
                class_names = ["negative", "positive", "neutral"]
                loader_module = "dataset_ro"
                preprocess_module = "preprocessing_ro"
                preprocess_fn = "preprocess_batch"
            elif "RED" in pipeline_dir_str or "Emotion" in pipeline_dir_str:
                dataset_name = "red"
                text_col = "text"
                label_col = "label"
                class_names = ["sadness", "surprise", "fear", "anger", "neutral", "trust", "joy"]
                loader_module = "dataset_red"
                preprocess_module = "preprocessing_red"
                preprocess_fn = "preprocess_batch_red"
            else:
                dataset_name = "imdb"
                text_col = "review"
                label_col = "label"
                class_names = ["negative", "positive"]
                loader_module = "dataset_en"
                preprocess_module = "preprocessing_en"
                preprocess_fn = "preprocess_batch"
                
            registry[model_name] = {
                "format": "summary",
                "pipeline_dir": pipeline_dir,
                "results_dir": results_dir,
                "dataset_name": dataset_name,
                "text_col": text_col,
                "label_col": label_col,
                "class_names": class_names,
                "loader_module": loader_module,
                "preprocess_module": preprocess_module,
                "preprocess_fn": preprocess_fn
            }
            
    # 2. Discover Format B models (pre-rendered images and iframe LIME HTML)
    for path in root_path.rglob("*"):
        if path.is_dir() and (path.name in ["Results", "output_WELFake", "output_ita_bert", "output_umberto"] or path.name.startswith("output_")):
            test_file = path / "lime_explanation_0.png"
            if test_file.exists():
                pipeline_dir = path.parent
                if path.name.startswith("output_"):
                    model_name = f"{pipeline_dir.name} ({path.name})"
                else:
                    model_name = pipeline_dir.name
                    
                if model_name in registry:
                    continue
                    
                count = sum(1 for f in path.glob("lime_explanation_*.png"))
                
                registry[model_name] = {
                    "format": "images",
                    "pipeline_dir": pipeline_dir,
                    "results_dir": path,
                    "count": count
                }
                
    return registry

MODEL_REGISTRY = discover_model_registry(project_root)
print(f"Discovered {len(MODEL_REGISTRY)} models with explainability results:")
for m in sorted(MODEL_REGISTRY.keys()):
    fmt = MODEL_REGISTRY[m]["format"].upper()
    info_str = f"Dataset: {MODEL_REGISTRY[m].get('dataset_name', 'N/A')}" if fmt == "SUMMARY" else f"Count: {MODEL_REGISTRY[m]['count']} static samples"
    print(f"  • {m:<45} | Format: {fmt:<8} | {info_str}")

Discovered 12 models with explainability results:
  • 1_BERT_SA_IMDB                                | Format: IMAGES   | Count: 5 static samples
  • 1_BERT_Sentiment_Articles                     | Format: SUMMARY  | Dataset: ro_articles
  • 1_ITA_BERT_SA_FEEL-IT (output_ita_bert)       | Format: IMAGES   | Count: 5 static samples
  • 2_DistilBERT_SA_IMDB                          | Format: IMAGES   | Count: 5 static samples
  • 2_RoBERT_Emotion_RED                          | Format: SUMMARY  | Dataset: red
  • 2_UmBERTo_SA_FEEL-IT (output_umberto)         | Format: IMAGES   | Count: 5 static samples
  • 3_BERT_WELFake (output_WELFake)               | Format: IMAGES   | Count: 5 static samples
  • 3_RoBERT_Conv_Emotion_RED                     | Format: SUMMARY  | Dataset: red
  • 4_DistilBERT_NEWS (output_WELFake)            | Format: IMAGES   | Count: 5 static samples
  • 4_RoBERT_Ensemble_Emotion_RED                 | Format: SUMMARY  | Dataset: red
  • 5_DistilBERT_SA_Practical       

In [3]:
DATASET_CACHE = {}

def get_dataset(dataset_name, model_info):
    """
    Loads the raw test split dataframe from the corresponding dataset module and caches it.
    """
    if dataset_name in DATASET_CACHE:
        return DATASET_CACHE[dataset_name]
        
    loader_name = model_info["loader_module"]
    loader_mod = __import__(loader_name)
    
    if dataset_name == "red":
        _, _, df_test = loader_mod.load_splits("classification")
    else:
        _, _, df_test = loader_mod.load_splits()
        
    DATASET_CACHE[dataset_name] = df_test
    return df_test

def preprocess_single_text(text, model_info):
    """
    Applies the corresponding preprocessing pipeline to a single raw text on-the-fly.
    """
    prep_mod_name = model_info["preprocess_module"]
    fn_name = model_info["preprocess_fn"]
    
    prep_mod = __import__(prep_mod_name)
    preprocess_fn = getattr(prep_mod, fn_name)
    
    res = preprocess_fn([text])
    if isinstance(res, tuple):
        cleaned = res[0]
    else:
        cleaned = res
        
    return cleaned[0]

In [4]:
def highlight_lime_text(text, top_words):
    """
    Highlights words in preprocessed text based on LIME feature importances.
    """
    if not top_words:
        return text
    word_weights = {w.lower(): wt for w, wt in top_words}
    
    # Split keeping non-alphanumeric delimiters
    tokens = re.split(r'(\W+)', text)
    highlighted = []
    for token in tokens:
        token_lower = token.lower()
        if token_lower in word_weights:
            weight = word_weights[token_lower]
            # Highlight color coding: light green for positive contribution, light red for negative
            color = "rgba(40, 167, 69, 0.25)"
            if weight < 0:
                color = "rgba(220, 53, 69, 0.25)"
            border_color = "rgba(40, 167, 69, 0.5)"
            if weight < 0:
                border_color = "rgba(220, 53, 69, 0.5)"
            text_color = "#155724"
            if weight < 0:
                text_color = "#721c24"
            tooltip = f"LIME weight: {weight:+.4f}"
            highlighted.append(
                f'<span style="background-color: {color}; border: 1px solid {border_color}; color: {text_color}; '
                f'padding: 1px 3px; border-radius: 3px; font-weight: 500; cursor: help;" title="{tooltip}">{token}</span>'
            )
        else:
            highlighted.append(token)
    return "".join(highlighted)

def highlight_shap_text(text, top_tokens):
    """
    Highlights words in preprocessed text based on cleaned SHAP token importances.
    """
    if not top_tokens:
        return text
    token_weights = {}
    for tok, wt in top_tokens:
        clean_tok = tok.strip().strip(",.-!?;:\"'()[]{}").lower()
        if clean_tok:
            token_weights[clean_tok] = wt
            
    tokens = re.split(r'(\W+)', text)
    highlighted = []
    for token in tokens:
        token_lower = token.lower()
        if token_lower in token_weights:
            weight = token_weights[token_lower]
            # Highlight color coding: light blue for positive contribution, orange/yellow for negative
            color = "rgba(0, 123, 255, 0.22)"
            if weight < 0:
                color = "rgba(255, 193, 7, 0.25)"
            border_color = "rgba(0, 123, 255, 0.45)"
            if weight < 0:
                border_color = "rgba(255, 193, 7, 0.5)"
            text_color = "#004085"
            if weight < 0:
                text_color = "#856404"
            tooltip = f"SHAP value: {weight:+.4f}"
            highlighted.append(
                f'<span style="background-color: {color}; border: 1px solid {border_color}; color: {text_color}; '
                f'padding: 1px 3px; border-radius: 3px; font-weight: 500; cursor: help;" title="{tooltip}">{token}</span>'
            )
        else:
            highlighted.append(token)
    return "".join(highlighted)

In [5]:
def plot_importances(lime_words, shap_tokens):
    """
    Renders LIME and SHAP top 10 feature importances side-by-side as horizontal bar charts.
    """
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    # 1. LIME Plot
    if lime_words:
        words, weights = zip(*lime_words)
        sorted_idx = np.argsort(np.abs(weights))
        words = [words[i] for i in sorted_idx]
        weights = [weights[i] for i in sorted_idx]
        
        colors = ['#28a745' if w > 0 else '#dc3545' for w in weights]
        ax1.barh(words, weights, color=colors, alpha=0.85)
        ax1.set_title("LIME Top Feature Importance", fontsize=12, fontweight='bold', pad=12)
        ax1.axvline(0, color='gray', linestyle='--', linewidth=0.8)
        ax1.spines['top'].set_visible(False)
        ax1.spines['right'].set_visible(False)
        ax1.spines['left'].set_color('#cbd5e1')
        ax1.spines['bottom'].set_color('#cbd5e1')
        ax1.grid(axis='x', linestyle=':', alpha=0.5)
    else:
        ax1.text(0.5, 0.5, "No LIME data available", ha='center', va='center', fontsize=12)
        ax1.axis('off')
        
    # 2. SHAP Plot
    if shap_tokens:
        tokens, weights = zip(*shap_tokens)
        sorted_idx = np.argsort(np.abs(weights))
        tokens = [tokens[i] for i in sorted_idx]
        weights = [weights[i] for i in sorted_idx]
        
        colors = ['#007bff' if w > 0 else '#ffc107' for w in weights]
        ax2.barh(tokens, weights, color=colors, alpha=0.85)
        ax2.set_title("SHAP Top Token Importance", fontsize=12, fontweight='bold', pad=12)
        ax2.axvline(0, color='gray', linestyle='--', linewidth=0.8)
        ax2.spines['top'].set_visible(False)
        ax2.spines['right'].set_visible(False)
        ax2.spines['left'].set_color('#cbd5e1')
        ax2.spines['bottom'].set_color('#cbd5e1')
        ax2.grid(axis='x', linestyle=':', alpha=0.5)
    else:
        ax2.text(0.5, 0.5, "No SHAP data available", ha='center', va='center', fontsize=12)
        ax2.axis('off')
        
    plt.tight_layout()
    plt.show()

def show_explanation(model_name, sample_idx):
    """
    Main rendering controller for a selected model and sample index.
    Loads LIME/SHAP metrics, aligns example indices, highlights text, plots importances, and embeds the HTML report.
    """
    if not model_name:
        display(HTML("<div style='color: orange;'>Please select a model.</div>"))
        return
        
    model_info = MODEL_REGISTRY[model_name]
    results_dir = Path(model_info["results_dir"])
    
    # --- RENDER PATH FOR STATIC IMAGE-BASED RUNS (Format B) ---
    if model_info["format"] == "images":
        lime_img = results_dir / f"lime_explanation_{sample_idx}.png"
        shap_img = results_dir / f"shap_explanation_{sample_idx}.png"
        lime_html_path = results_dir / f"lime_explanation_{sample_idx}.html"
        
        rel_lime_img = os.path.relpath(lime_img, start=notebook_dir)
        rel_shap_img = os.path.relpath(shap_img, start=notebook_dir)
        
        header_html = f"""
        <div style='font-family: system-ui, -apple-system, sans-serif; background: #ffffff; border: 1px solid #e2e8f0; border-radius: 8px; padding: 15px; margin-bottom: 20px; box-shadow: 0 1px 3px rgba(0,0,0,0.05);'>
            <h3 style='margin-top: 0; margin-bottom: 8px; color: #0f172a; border-bottom: 2px solid #f1f5f9; padding-bottom: 8px; font-size: 16px;'>
                🖼 Static Image Explanation Dashboard
            </h3>
            <div style='display: flex; gap: 20px; font-size: 14px;'>
                <div><strong>Model:</strong> <code style='background: #f1f5f9; padding: 2px 6px; border-radius: 4px; color: #0f172a;'>{model_name}</code></div>
                <div><strong>Sample Index:</strong> <code style='background: #f1f5f9; padding: 2px 6px; border-radius: 4px; color: #0f172a;'>{sample_idx}</code></div>
            </div>
        </div>
        """
        display(HTML(header_html))
        
        # Use single quotes for HTML attributes to avoid JSON string boundary errors
        images_html = f"""
        <div style='display: flex; gap: 20px; justify-content: center; margin-bottom: 20px; font-family: system-ui, -apple-system, sans-serif;'>
            <div style='flex: 1; text-align: center; border: 1px solid #cbd5e1; border-radius: 6px; padding: 10px; background: #ffffff;'>
                <h4 style='margin-top: 0; color: #16a34a; font-weight: 600; margin-bottom: 10px;'>🟢 LIME Explanation</h4>
                <img src='{rel_lime_img}' style='max-width: 100%; border-radius: 4px; border: 1px solid #e2e8f0;' onerror="this.style.display='none'; this.nextElementSibling.style.display='block';" />
                <div style='display:none; color: orange; font-style: italic; padding: 20px;'>LIME image not found</div>
            </div>
            <div style='flex: 1; text-align: center; border: 1px solid #cbd5e1; border-radius: 6px; padding: 10px; background: #ffffff;'>
                <h4 style='margin-top: 0; color: #2563eb; font-weight: 600; margin-bottom: 10px;'>🔵 SHAP Explanation</h4>
                <img src='{rel_shap_img}' style='max-width: 100%; border-radius: 4px; border: 1px solid #e2e8f0;' onerror="this.style.display='none'; this.nextElementSibling.style.display='block';" />
                <div style='display:none; color: orange; font-style: italic; padding: 20px;'>SHAP image not found</div>
            </div>
        </div>
        """
        display(HTML(images_html))
        
        try:
            if lime_html_path.exists():
                with open(lime_html_path, "r", encoding="utf-8") as f:
                    html_content = f.read()
                escaped_html = html.escape(html_content)
                iframe_html = "<div style='font-family: system-ui, -apple-system, sans-serif; margin-top: 20px;'><h4 style='color: #475569; margin-bottom: 8px; font-weight: 600;'>🖼 LIME Interactive HTML Report:</h4><iframe srcdoc='" + escaped_html + "' width='100%' height='480px' style='border: 1px solid #cbd5e1; border-radius: 6px;'></iframe></div>"
                display(HTML(iframe_html))
            else:
                display(HTML(f"<div style='color: orange; font-style: italic;'>LIME HTML file not found at: {lime_html_path}</div>"))
        except Exception as e:
            display(HTML(f"<div style='color: red;'>Error loading LIME HTML file: {e}</div>"))
        return

    # --- RENDER PATH FOR SUMMARY-BASED RUNS (Format A) ---
    lime_sum_path = results_dir / "lime_explanations" / "lime_summary.json"
    shap_sum_path = results_dir / "shap_explanations" / "shap_summary.json"
    
    if not lime_sum_path.exists() or not shap_sum_path.exists():
        display(HTML(f"<div style='color: red; font-weight: bold;'>Error: Explanation summary files not found under {results_dir}.</div>"))
        return
        
    with open(lime_sum_path, "r", encoding="utf-8") as f:
        lime_data = json.load(f)
    with open(shap_sum_path, "r", encoding="utf-8") as f:
        shap_data = json.load(f)
        
    if sample_idx >= len(lime_data) or sample_idx >= len(shap_data):
        display(HTML("<div style='color: red;'>Sample index out of bounds.</div>"))
        return
        
    lime_entry = lime_data[sample_idx]
    shap_entry = shap_data[sample_idx]
    
    if lime_entry["example_idx"] != shap_entry["example_idx"]:
        display(HTML(f"<div style='color: orange; margin-bottom: 10px;'>⚠️ Warning: LIME and SHAP examples are misaligned! LIME index: {lime_entry['example_idx']}, SHAP index: {shap_entry['example_idx']}</div>"))
        
    example_idx = lime_entry["example_idx"]
    
    # Load raw text from dataset split and resolve true label
    try:
        df_test = get_dataset(model_info["dataset_name"], model_info)
        raw_text = df_test.iloc[example_idx][model_info["text_col"]]
        
        # Address KeyErrors for RED datasets (multilabel index argmax lookup)
        if model_info["dataset_name"] == "red":
            agreed_cols = [f"agreed_label_{e}" for e in model_info["class_names"]]
            row_labels = df_test.iloc[example_idx][agreed_cols].values
            true_label_val = row_labels.argmax()
            true_label_name = model_info["class_names"][int(true_label_val)]
        else:
            true_label_val = df_test.iloc[example_idx][model_info["label_col"]]
            if isinstance(true_label_val, (int, float, np.integer)):
                true_label_name = model_info["class_names"][int(true_label_val)]
            else:
                true_label_name = str(true_label_val)
    except Exception as e:
        display(HTML(f"<div style='color: red; margin-bottom: 10px;'>Error loading text for index {example_idx} from dataset: {e}</div>"))
        raw_text = "[Error loading raw text]"
        true_label_name = lime_entry.get("true_label", "Unknown")
        
    # Preprocess text on-the-fly
    try:
        prep_text = preprocess_single_text(raw_text, model_info)
    except Exception as e:
        display(HTML(f"<div style='color: red; margin-bottom: 10px;'>Error preprocessing text: {e}</div>"))
        prep_text = raw_text
        
    # Generate Highlights
    lime_highlighted = highlight_lime_text(prep_text, lime_entry.get("top_words", []))
    shap_highlighted = highlight_shap_text(prep_text, shap_entry.get("top_tokens", []))
    
    # Render Dashboard Header
    correct = lime_entry.get("correct", True)
    badge_color = "#16a34a" if correct else "#dc2626"
    badge_text = "Correct ✓" if correct else "Incorrect ✗"
    
    meta_html = f"""
    <div style='font-family: system-ui, -apple-system, sans-serif; background: #ffffff; border: 1px solid #e2e8f0; border-radius: 8px; padding: 15px; margin-bottom: 20px; box-shadow: 0 1px 3px rgba(0,0,0,0.05);'>
        <h3 style='margin-top: 0; margin-bottom: 12px; color: #0f172a; border-bottom: 2px solid #f1f5f9; padding-bottom: 8px; font-size: 16px;'>
            📊 Model Run Explanation Dashboard
        </h3>
        <div style='display: flex; gap: 20px; flex-wrap: wrap; align-items: center; font-size: 14px;'>
            <div><strong>Model:</strong> <code style='background: #f1f5f9; padding: 2px 6px; border-radius: 4px; color: #0f172a;'>{model_name}</code></div>
            <div><strong>Dataset Index:</strong> <code style='background: #f1f5f9; padding: 2px 6px; border-radius: 4px; color: #0f172a;'>{example_idx}</code></div>
            <div><strong>True Label:</strong> <span style='background: #e2e8f0; padding: 2px 8px; border-radius: 20px; font-weight: 500; font-size: 13px; color: #334155;'>{true_label_name}</span></div>
            <div><strong>Predicted Label:</strong> <span style='background: #e2e8f0; padding: 2px 8px; border-radius: 20px; font-weight: 500; font-size: 13px; color: #334155;'>{lime_entry.get('predicted_label', 'Unknown')}</span></div>
            <div><span style='background-color: {badge_color}; color: white; padding: 4px 10px; border-radius: 20px; font-weight: bold; font-size: 13px; display: inline-block;'>{badge_text}</span></div>
        </div>
    </div>
    """
    display(HTML(meta_html))
    
    # Render Collapsible Dropdowns
    raw_and_prep_html = f"""
    <div style='font-family: system-ui, -apple-system, sans-serif;'>
        <!-- Collapsible for Raw Text -->
        <details style='margin-bottom: 12px; border: 1px solid #cbd5e1; border-radius: 6px; background-color: #f8fafc;'>
            <summary style='padding: 12px 16px; font-weight: 600; cursor: pointer; outline: none; user-select: none; color: #334155; list-style: none; display: flex; justify-content: space-between; align-items: center;'>
                <span>📋 Raw Text (Before Preprocessing)</span>
                <span style='font-size: 12px; color: #64748b;'>▶ Click to Expand / Collapse</span>
            </summary>
            <div style='padding: 16px; background-color: #ffffff; border-top: 1px solid #cbd5e1; border-bottom-left-radius: 6px; border-bottom-right-radius: 6px; font-family: monospace; font-size: 14px; white-space: pre-wrap; line-height: 1.5; color: #0f172a;'>{raw_text}</div>
        </details>
        
        <!-- Collapsible for Highlighted Preprocessed Text -->
        <details open style='margin-bottom: 20px; border: 1px solid #cbd5e1; border-radius: 6px; background-color: #f8fafc;'>
            <summary style='padding: 12px 16px; font-weight: 600; cursor: pointer; outline: none; user-select: none; color: #334155; list-style: none; display: flex; justify-content: space-between; align-items: center;'>
                <span>🔍 Highlighted Preprocessed Text (Explainer Inputs)</span>
                <span style='font-size: 12px; color: #64748b;'>▼ Expanded by Default</span>
            </summary>
            <div style='padding: 16px; background-color: #ffffff; border-top: 1px solid #cbd5e1; border-bottom-left-radius: 6px; border-bottom-right-radius: 6px; font-size: 15px; line-height: 1.6; color: #0f172a;'>
                <div style='margin-bottom: 20px;'>
                    <h4 style='margin-top: 0; margin-bottom: 8px; color: #16a34a; display: flex; align-items: center; gap: 6px; font-weight: 600;'>
                        🟢 LIME Feature Importance Highlights:
                    </h4>
                    <div style='padding: 12px; border: 1px solid #e2e8f0; border-radius: 6px; background-color: #fafafa; font-family: system-ui, -apple-system, sans-serif;'>
                        {lime_highlighted}
                    </div>
                </div>
                <div>
                    <h4 style='margin-top: 0; margin-bottom: 8px; color: #2563eb; display: flex; align-items: center; gap: 6px; font-weight: 600;'>
                        🔵 SHAP Feature Importance Highlights:
                    </h4>
                    <div style='padding: 12px; border: 1px solid #e2e8f0; border-radius: 6px; background-color: #fafafa; font-family: system-ui, -apple-system, sans-serif;'>
                        {shap_highlighted}
                    </div>
                </div>
            </div>
        </details>
    </div>
    """
    display(HTML(raw_and_prep_html))
    
    # Render Matplotlib Bar Charts
    plot_importances(lime_entry.get("top_words", []), shap_entry.get("top_tokens", []))
    
    # Render LIME Interactive HTML Report
    try:
        html_file = lime_entry["html_file"]
        html_path = results_dir / "lime_explanations" / html_file
        if html_path.exists():
            with open(html_path, "r", encoding="utf-8") as f:
                html_content = f.read()
            escaped_html = html.escape(html_content)
            iframe_html = "<div style='font-family: system-ui, -apple-system, sans-serif; margin-top: 20px;'><h4 style='color: #475569; margin-bottom: 8px; font-weight: 600;'>🖼 LIME Interactive HTML Report:</h4><iframe srcdoc='" + escaped_html + "' width='100%' height='480px' style='border: 1px solid #cbd5e1; border-radius: 6px;'></iframe></div>"
            display(HTML(iframe_html))
        else:
            display(HTML(f"<div style='color: orange; font-style: italic;'>LIME HTML file not found at: {html_path}</div>"))
    except Exception as e:
        display(HTML(f"<div style='color: red;'>Error loading LIME HTML file: {e}</div>"))

In [6]:
# Create interactive widgets
model_dropdown = widgets.Dropdown(
    options=sorted(list(MODEL_REGISTRY.keys())),
    value=sorted(list(MODEL_REGISTRY.keys()))[0] if MODEL_REGISTRY else None,
    description='Selected Model:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='380px')
)

sample_slider = widgets.IntSlider(
    value=0,
    min=0,
    max=0,
    description='Sample Index:',
    continuous_update=False,
    layout=widgets.Layout(width='400px')
)

sample_text = widgets.BoundedIntText(
    value=0,
    min=0,
    max=0,
    description='Or Type Index:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='180px')
)

# Link slider and textbox
widgets.jslink((sample_slider, 'value'), (sample_text, 'value'))

def on_model_change(change):
    model_name = change['new']
    if not model_name:
        return
        
    model_info = MODEL_REGISTRY[model_name]
    
    if model_info.get("format") == "images":
        count = model_info["count"]
    else:
        results_dir = Path(model_info["results_dir"])
        lime_sum_path = results_dir / "lime_explanations" / "lime_summary.json"
        
        if lime_sum_path.exists():
            try:
                with open(lime_sum_path, "r", encoding="utf-8") as f:
                    data = json.load(f)
                    count = len(data)
            except Exception:
                count = 1
        else:
            count = 1
            
    val_max = max(0, count - 1)
    sample_slider.max = val_max
    sample_text.max = val_max
    sample_slider.value = 0
    sample_text.value = 0

model_dropdown.observe(on_model_change, names='value')

# Display panel layout
ui = widgets.HBox([model_dropdown, sample_slider, sample_text])
out = widgets.interactive_output(
    show_explanation,
    {'model_name': model_dropdown, 'sample_idx': sample_slider}
)

display(ui, out)

# Trigger initial setup
if MODEL_REGISTRY:
    on_model_change({'new': model_dropdown.value})

Output()